# Capítulo 1 — Ambiente, dados e primeiros passos

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/01_ambiente_e_dados.ipynb)

---


## 1.1 Motivação

Boa parte do trabalho em agrometeorologia — cálculo de balanço hídrico, evapotranspiração,
zoneamento de risco climático — é **repetitivo e depende de séries de dados longas**: dezenas
de estações, décadas de observações, centenas de municípios. Fazer isso "na mão" (planilha por
planilha) não escala.

Este curso ensina a resolver esses mesmos problemas — os mesmos que você já resolve na
apostila da disciplina FT1031 — de forma **automatizada e reprodutível**, usando Python.

Ao final deste capítulo você terá:

- um ambiente de trabalho pronto (Google Colab ou Jupyter local);
- a biblioteca [`agrometeorologiapy`](https://github.com/fcoliveira-utfpr/agrometeorologiapy) instalada;
- noção de onde buscar dados climáticos reais (INMET, NASA POWER, TerraClimate);
- um primeiro DataFrame com dados climáticos organizado e pronto para os próximos capítulos.


## 1.2 Ambiente de trabalho: notebooks

Um notebook (este arquivo) mistura **texto explicativo** (como esta célula, em Markdown) com
**código executável** (a próxima célula, em Python). Cada capítulo do curso é um notebook
independente — você pode abrir só o que precisa, sem carregar o curso inteiro.

Regras práticas para não ter dor de cabeça com notebooks:

1. **Execute as células em ordem**, de cima para baixo, pelo menos na primeira vez.
2. Se algo parecer "bugado" (uma variável com valor errado, um erro estranho), use
   `Ambiente de execução → Reiniciar sessão` (Colab) ou `Kernel → Restart` (Jupyter) e rode
   tudo de novo. Isso limpa o estado e evita efeitos fantasma de execuções fora de ordem.
3. Célula de código só mostra resultado do que está na **última linha** (ou de `print(...)`
   explícito) — isso vai aparecer bastante nos exemplos abaixo.


In [ ]:
# Célula de teste — se isto rodar sem erro, o ambiente está funcionando.
import sys
print(f"Python {sys.version.split()[0]} rodando normalmente.")


## 1.3 Instalando a `agrometeorologiapy`

A [`agrometeorologiapy`](https://github.com/fcoliveira-utfpr/agrometeorologiapy) reúne as
fórmulas que vamos usar do Capítulo 2 em diante: radiação solar, temperatura, umidade,
balanço de energia, evapotranspiração (Thornthwaite, Camargo-Maluf, Hargreaves-Samani,
Priestley-Taylor, Penman-Monteith FAO-56), graus-dia e balanço hídrico.

A documentação matemática de cada função (fórmula original, variáveis, unidades) está em
[`docs/FORMULAS.md`](https://github.com/fcoliveira-utfpr/agrometeorologiapy/blob/main/docs/FORMULAS.md)
— vale manter essa página aberta como referência ao longo do curso.


In [ ]:
%pip install -q agrometeorologiapy pandas requests matplotlib


In [ ]:
import agrometeorologiapy as amp

# Exemplo mínimo: número do dia do ano e fotoperíodo para uma data e latitude
nda = amp.nda(dia=15, mes=1)                                 # 15 de janeiro
declinacao = amp.declinacao_solar(nda)
Hn = amp.angulo_horario_nascer(lat=-24.86, declinacao=declinacao)  # Santa Helena-PR
N = amp.fotoperiodo(Hn)

print(f"NDA: {nda}")
print(f"Declinação solar: {declinacao:.2f} graus")
print(f"Fotoperíodo em 15/jan para Santa Helena-PR: {N:.2f} horas")


Se a célula acima rodou e imprimiu três valores, a biblioteca está instalada e funcionando.
Vamos entender essas funções (e as fórmulas por trás delas) em detalhe no **Capítulo 2**.


## 1.4 Pandas essencial para séries climáticas

Praticamente todo dado agrometeorológico é uma **série temporal**: uma coluna de datas e uma
ou mais colunas de variáveis (temperatura, precipitação, radiação...). O `pandas` é a
ferramenta padrão em Python para isso. Três operações que vamos repetir o curso inteiro:

1. **Criar um índice de datas** (`pd.date_range`)
2. **Agregar por período** (`resample`) — por exemplo, transformar dados diários em totais
   mensais de chuva ou médias mensais de temperatura
3. **Selecionar por período** (fatiar por ano, mês, intervalo de datas)


In [ ]:
import pandas as pd
import numpy as np

# Série sintética só para praticar a mecânica do pandas (dados reais entram na próxima seção)
datas = pd.date_range("2024-01-01", "2024-12-31", freq="D")
precipitacao_mm = np.random.default_rng(42).gamma(shape=1.0, scale=6.0, size=len(datas))

df_exemplo = pd.DataFrame({"data": datas, "precipitacao_mm": precipitacao_mm})
df_exemplo = df_exemplo.set_index("data")

# Agregação mensal: soma de precipitação por mês
chuva_mensal = df_exemplo["precipitacao_mm"].resample("ME").sum()
chuva_mensal.round(1)


In [ ]:
# Selecionar um intervalo específico (ex.: só o trimestre de inverno)
df_exemplo.loc["2024-06-01":"2024-08-31"].describe()


## 1.5 Fontes de dados climáticos operacionais

Em produção (fora de exercícios didáticos), os dados vêm de fontes externas. As três que mais
vamos usar neste curso:

| Fonte | O que oferece | Quando usar |
|---|---|---|
| **INMET** (Instituto Nacional de Meteorologia) | Estações automáticas e convencionais no Brasil, dados horários/diários | Quando você precisa de uma estação específica, com histórico oficial brasileiro |
| **NASA POWER** | Reanálise/satélite, grade global, acesso via API REST simples, sem necessidade de cadastro | Ótimo para começar rápido e para qualquer ponto do globo, inclusive onde não há estação |
| **TerraClimate / Google Earth Engine** | Grades históricas globais (1958-presente), acesso via GEE | Quando o problema é em escala municipal/regional, como no zoneamento do Capítulo 7 |

Neste capítulo vamos usar a **NASA POWER**, porque não exige autenticação e já entrega o dado
pronto em JSON — ideal para o primeiro contato.


## 1.6 Atividade guiada — baixando dados reais da NASA POWER

Vamos baixar uma série diária de temperatura, precipitação e radiação para **Santa Helena-PR**
(latitude -24,86°, longitude -54,33°) e organizar em um DataFrame.

A API da NASA POWER (endpoint `temporal/daily/point`) recebe: lista de parâmetros, coordenadas,
datas de início/fim e formato de saída. Documentação completa em
<https://power.larc.nasa.gov/docs/services/api/>.


In [ ]:
import requests

LAT, LON = -24.86, -54.33          # Santa Helena - PR
INICIO, FIM = "20230101", "20231231"

parametros = [
    "T2M",              # Temperatura média a 2 m (°C)
    "T2M_MAX",          # Temperatura máxima a 2 m (°C)
    "T2M_MIN",          # Temperatura mínima a 2 m (°C)
    "PRECTOTCORR",      # Precipitação corrigida (mm/dia)
    "ALLSKY_SFC_SW_DWN",# Radiação solar incidente (MJ/m²/dia)
    "RH2M",             # Umidade relativa a 2 m (%)
    "WS2M",             # Velocidade do vento a 2 m (m/s)
]

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": ",".join(parametros),
    "community": "AG",
    "longitude": LON,
    "latitude": LAT,
    "start": INICIO,
    "end": FIM,
    "format": "JSON",
}

resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()
dados_json = resposta.json()

print("Requisição concluída com status:", resposta.status_code)


In [ ]:
# A API devolve um dicionário aninhado; cada parâmetro é uma série {data: valor}.
# Vamos organizar isso em um DataFrame com uma linha por dia.

propriedades = dados_json["properties"]["parameter"]

df_clima = pd.DataFrame(propriedades)
df_clima.index = pd.to_datetime(df_clima.index, format="%Y%m%d")
df_clima.index.name = "data"

# A NASA POWER usa -999 como código de dado ausente
df_clima = df_clima.replace(-999, np.nan)

df_clima.head()


In [ ]:
# Visão geral rápida da série baixada
df_clima.describe().round(2)


In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

eixos[0].plot(df_clima.index, df_clima["T2M_MAX"], label="T máx", color="firebrick")
eixos[0].plot(df_clima.index, df_clima["T2M_MIN"], label="T mín", color="steelblue")
eixos[0].set_ylabel("Temperatura (°C)")
eixos[0].legend()
eixos[0].set_title("Santa Helena-PR — 2023 (dados NASA POWER)")

eixos[1].bar(df_clima.index, df_clima["PRECTOTCORR"], color="teal", width=1.0)
eixos[1].set_ylabel("Precipitação (mm/dia)")

plt.tight_layout()
plt.show()


Guarde esse `df_clima` — vamos reaproveitar exatamente essa estrutura (data no índice, uma
coluna por variável) em todos os capítulos seguintes.


## 1.7 Desafio

1. Repita o download da NASA POWER trocando `LAT`/`LON` pela sua cidade de interesse (ex.:
   Cascavel-PR: -24,95, -53,46) e o intervalo de datas por um ano completo diferente.
2. Calcule, a partir do `df_clima`, a **precipitação total anual** e o **número de dias com
   chuva acima de 20 mm** (considerado "chuva forte" pela classificação da apostila).
3. *(opcional)* Compare a temperatura média mensal do seu município com a de Santa Helena-PR
   no mesmo ano, em um único gráfico.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 1.8 Checkpoint

Antes de seguir para o **Capítulo 2 — Radiação solar e fotoperíodo**, você deve ter:

- [ ] rodado um notebook do início ao fim sem erros;
- [ ] a `agrometeorologiapy` instalada e importada com sucesso;
- [ ] um `df_clima` com pelo menos um ano de dados diários reais (T2M, T2M_MAX, T2M_MIN,
      PRECTOTCORR, ALLSKY_SFC_SW_DWN, RH2M, WS2M), indexado por data;
- [ ] resolvido pelo menos o item 1 do desafio.

No próximo capítulo vamos usar exatamente as colunas `ALLSKY_SFC_SW_DWN` (radiação) e as
coordenadas geográficas para reproduzir, em código, o exercício de radiação solar de
Cascavel-PR da apostila — e depois generalizar para qualquer local e data.
